# SAD-VREAL Pipeline

Pipeline phát hiện quảng cáo BĐS đáng ngờ trên mogi.vn bằng Weak Supervision.

**Chạy theo thứ tự:** STEP 1 → STEP 2 → STEP 3

**File cần có:**
- `mogi_clean.csv` — dữ liệu thô từ mogi.vn
- `price_ref_hcm_hn_mogi.csv` — bảng giá tham chiếu 51 quận/huyện

---
## STEP 1 — Weak Supervision Labeling

Tiền xử lý dữ liệu và gắn nhãn proxy bằng 6 tín hiệu heuristic.  
Không cần nhãn chuyên gia.

**Output:** `mogi_labeled.csv` · `mogi_model_input.csv`

In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
from unidecode import unidecode

# ══════════════════════════════════════════════════════════════════
# CẤU HÌNH — chỉnh đường dẫn nếu cần
# ══════════════════════════════════════════════════════════════════
INPUT_CSV    = "mogi_clean.csv"
PRICE_REF    = "price_ref_hcm_hn_mogi.csv"
OUTPUT_FULL  = "mogi_labeled.csv"          # toàn bộ cột + label
OUTPUT_MODEL = "mogi_model_input.csv"      # chỉ cột cần cho STEP2

# Ngưỡng r(x) — tự động tính từ phân phối thực nghiệm của dataset đầu vào.
# Không hardcode giá trị tuyệt đối; thay vào đó giữ cố định PERCENTILE.
# Hai hằng số dưới đây là PERCENTILE (0–100), không phải giá trị r(x).
# Giá trị tuyệt đối THR_SEVERE / THR_VIOLATION được tính tại BƯỚC 5
# sau khi r(x) đã được tính trên toàn dataset.
#
#   PCTL_SEVERE    = 90  → ~10% ads lệch giá cực đoan nhất → hard signal
#   PCTL_VIOLATION = 82  → ~8% kế tiếp → soft signal
#
# Ý nghĩa: giá trị tuyệt đối thay đổi theo dataset (vd: 2.5 trên Chợ Tốt,
# 2.65 trên mogi) nhưng selectivity (top 10% / top 8%) luôn nhất quán.
PCTL_SEVERE    = 90   # percentile cho hard signal (price_band_severe)
PCTL_VIOLATION = 82   # percentile cho soft signal (price_band_violation)
THR_SOFT_MIN   = 5    # tổng điểm soft để label suspicious

# Hệ số nới lỏng cho căn hộ cao cấp (premium apartment)
PREMIUM_RELAX = 1.3

# ══════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════

def norm(s):
    """Lowercase + remove diacritics + collapse spaces."""
    if pd.isna(s):
        return ""
    return re.sub(r"\s+", " ", unidecode(str(s).lower())).strip()


def dist_key(d):
    """Strip prefix quan/huyen/thi xa khỏi tên quận."""
    s = norm(d)
    return re.sub(r"^(quan|huyen|thi xa|thanh pho|tp)\s+", "", s).strip()


# ══════════════════════════════════════════════════════════════════
# BƯỚC 1 — Load data
# ══════════════════════════════════════════════════════════════════

print("=" * 60)
print("STEP1_mogi — Weak-supervision labeling pipeline")
print("=" * 60)

df  = pd.read_csv(INPUT_CSV, low_memory=False)
ref = pd.read_csv(PRICE_REF)

print(f"\n[1] Loaded: {len(df)} rows × {len(df.columns)} cols")
print(f"    Price ref: {len(ref)} districts")

# ══════════════════════════════════════════════════════════════════
# BƯỚC 2 — Preprocessing (dedup + price filter)
# ══════════════════════════════════════════════════════════════════

n0 = len(df)

# 2a. Exact duplicate: cùng full_text + price + area
df = df.drop_duplicates(
    subset=["full_text", "price_billion", "area_m2"], keep="first"
)
n1 = len(df)

# 2b. Near-duplicate: cùng description_clean + price + area
df = df.drop_duplicates(
    subset=["description_clean", "price_billion", "area_m2"], keep="first"
)
n2 = len(df)

# 2c. Loại bỏ giá/m² phi thực tế (< 1 hoặc > 2000 triệu/m²)
rho = df["price_per_m2_calc"]
df  = df[(rho >= 1) & (rho <= 2000)].copy().reset_index(drop=True)
n3  = len(df)

print(f"\n[2] Preprocessing: {n0} → {n1} → {n2} → {n3}")
print(f"    Removed: {n0-n1} exact dup | {n1-n2} near-dup | {n2-n3} bad price")

# ══════════════════════════════════════════════════════════════════
# BƯỚC 3 — Build price reference lookup
# ══════════════════════════════════════════════════════════════════

lookup = {}
for _, r in ref.iterrows():
    prov = norm(r["province"])
    dk   = dist_key(r["district"])
    lookup[(prov, dk)] = {
        "a":   float(r["apartment"]),
        "lnd": float(r["land"]),
        "h":   float(r["h"]),
        "f":   float(r["f"]),
        "m":   float(r["m"]),
    }

# Alias cho Quận 2, 9, Thủ Đức (ref có suffix "(TP. Thủ Đức)" trong tên)
for suffix, short in [
    ("2 (tp. thu duc)", "2"),
    ("9 (tp. thu duc)", "9"),
    ("thu duc (tp. thu duc)", "thu duc"),
]:
    key_src = ("tp ho chi minh", suffix)
    key_dst = ("tp ho chi minh", short)
    if key_src in lookup:
        lookup[key_dst] = lookup[key_src]

print(f"\n[3] Price lookup: {len(lookup)} keys")


def get_threshold(prov: str, dist: str):
    """
    Tra bảng giá cho (tỉnh, quận).
    Trả về (dict giá, nguồn): exact / fuzzy / province_mean / no_match
    """
    dk  = dist_key(dist)
    pv  = norm(prov)
    key = (pv, dk)
    if key in lookup:
        return lookup[key], "exact"
    # Fuzzy: so 8 ký tự đầu
    for (p, d), v in lookup.items():
        if p == pv and d[:8] == dk[:8] and len(dk) >= 4:
            return v, "fuzzy"
    # Fallback: trung bình cấp tỉnh
    prov_vals = [v for (p, _), v in lookup.items() if p == pv]
    if prov_vals:
        return (
            {k: float(np.mean([x[k] for x in prov_vals]))
             for k in ("a", "lnd", "h", "f", "m")},
            "province_mean",
        )
    return None, "no_match"

# ══════════════════════════════════════════════════════════════════
# BƯỚC 4 — Property classification từ house_type_norm
#
# Mogi đã cung cấp house_type rõ ràng cho từng loại BĐS.
# Mapping trực tiếp house_type_norm → _property_cat
# (không cần keyword heuristic như Chợ Tốt)
# ══════════════════════════════════════════════════════════════════

HT_TO_CAT = {
    # Nhà ở
    "nha mat pho, mat tien": "nha_mat_tien",
    "nha ngo, hem":          "nha_hem",
    "nha biet thu":          "nha_mat_tien",   # biệt thự → ngưỡng mặt tiền
    "nha pho lien ke":       "nha_mat_tien",   # liền kề → ngưỡng mặt tiền
    "shophouse":             "nha_mat_tien",   # shophouse → ngưỡng mặt tiền
    "mat bang kinh doanh":   "nha_mat_tien",   # mặt bằng → ngưỡng mặt tiền
    # Căn hộ
    "can ho chung cu":       "can_ho",
    # Đất
    "dat tho cu":            "dat",
    "dat nen du an":         "dat",
    "dat nong nghiep":       "dat",
    # Khác
    "nha tro, phong tro":    "nha_hem",        # nhà trọ → ngưỡng hẻm
    "van phong":             "other",
    "kho, xuong":            "other",
}

# Fallback cho 154 rows NaN house_type_norm
APT_KW  = re.compile(
    r"\b(can ho|chung cu|ccmn|officetel|penthouse|duplex)\b"
)
LAND_KW = re.compile(
    r"\b(dat nen|lo dat|dat tho cu|dat o |dat phan lo|"
    r"dat vuon|dat trong|dat nong nghiep|ban dat)\b"
)


def classify(row) -> str:
    ht = norm(str(row.get("house_type_norm", "")))

    # Direct mapping (96%+ rows)
    if ht and ht in HT_TO_CAT:
        return HT_TO_CAT[ht]

    # Fallback: keyword trong title + description
    text = norm(str(row.get("title_clean", ""))) + " " + \
           norm(str(row.get("full_text", ""))[:300])
    if APT_KW.search(text):
        return "can_ho"
    if LAND_KW.search(text):
        return "dat"

    # Cuối cùng: frontage_class
    fc = str(row.get("frontage_class", ""))
    if fc == "mat_tien":
        return "nha_mat_tien"
    if fc == "hem":
        return "nha_hem"

    return "nha_mat_tien"  # safe default


df["_property_cat"] = df.apply(classify, axis=1)

print(f"\n[4] Property classification:")
for cat, n in df["_property_cat"].value_counts().items():
    pct = n / len(df) * 100
    print(f"    {cat:<20} {n:>5}  ({pct:.1f}%)")

# ══════════════════════════════════════════════════════════════════
# BƯỚC 5 — Tính r(x) và u_hi / u_lo
# ══════════════════════════════════════════════════════════════════

u_hi_list, u_lo_list, src_list = [], [], []

for _, row in df.iterrows():
    thr, src = get_threshold(
        row.get("province_norm", ""),
        row.get("district_norm", ""),
    )
    if thr is None:
        u_hi_list.append(np.nan)
        u_lo_list.append(np.nan)
        src_list.append(src)
        continue

    # u_hi: ngưỡng trên theo loại BĐS
    u_hi_map = {
        "nha_mat_tien": thr["f"],
        "nha_hem":      thr["h"],
        "can_ho":       thr["a"],
        "dat":          thr["lnd"],
        "other":        thr["m"],
    }
    u_hi = u_hi_map[row["_property_cat"]]

    # is_can_ho_premium: nới lỏng ngưỡng 1.3× cho căn hộ diện tích > 80m²
    if row["_property_cat"] == "can_ho" and float(row.get("usable_area_num") or 0) > 80:
        u_hi = u_hi * PREMIUM_RELAX

    # u_lo: ngưỡng sàn = giá thấp nhất toàn quận (adaptive floor)
    u_lo = min(thr["a"], thr["lnd"], thr["h"], thr["f"])

    u_hi_list.append(u_hi)
    u_lo_list.append(u_lo)
    src_list.append(src)

df["_u_hi"]             = u_hi_list
df["_u_lo"]             = u_lo_list
df["_threshold_source"] = src_list

# Tính r(x) = max(ρ/u_hi, u_lo/ρ)
rho = df["price_per_m2_calc"].astype(float)
df["_r_typeaware"] = np.maximum(rho / df["_u_hi"], df["_u_lo"] / rho)

# ── Tự động tính ngưỡng từ phân phối r(x) của dataset này ──────────────
# Giữ cố định PERCENTILE (PCTL_SEVERE/PCTL_VIOLATION) thay vì giá trị
# tuyệt đối, để selectivity nhất quán bất kể phân phối r(x) của dataset.
# Kết quả: THR_SEVERE ≈ 2.5 trên Chợ Tốt, ≈ 2.65 trên mogi — khác nhau
# nhưng đều chọn ~top 10% / top 8% ads lệch giá nhất.
r_valid       = df["_r_typeaware"].dropna()
THR_SEVERE    = float(np.percentile(r_valid, PCTL_SEVERE))
THR_VIOLATION = float(np.percentile(r_valid, PCTL_VIOLATION))
# ────────────────────────────────────────────────────────────────────────

print(f"\n[5] r(x) computed:")
print(f"    District match — exact:        {(df['_threshold_source']=='exact').sum()}")
print(f"    District match — fuzzy:        {(df['_threshold_source']=='fuzzy').sum()}")
print(f"    District match — province_mean:{(df['_threshold_source']=='province_mean').sum()}")
print(f"    District match — no_match:     {(df['_threshold_source']=='no_match').sum()}")
r = df["_r_typeaware"].dropna()
print(f"    r(x): median={r.median():.3f} | "
      f"P{PCTL_VIOLATION}={r.quantile(PCTL_VIOLATION/100):.3f} | "
      f"P{PCTL_SEVERE}={r.quantile(PCTL_SEVERE/100):.3f}")
print(f"    THR_VIOLATION (P{PCTL_VIOLATION}) = {THR_VIOLATION:.4f}  [soft signal]")
print(f"    THR_SEVERE    (P{PCTL_SEVERE}) = {THR_SEVERE:.4f}  [hard signal]")

# ══════════════════════════════════════════════════════════════════
# BƯỚC 6 — Tính các signal flags
# ══════════════════════════════════════════════════════════════════

r = df["_r_typeaware"]

# Hard signals
df["flag_v4_price_band_severe"] = (r > THR_SEVERE).astype(int)

# Title–area mismatch: kích thước ghi trong title khác với area_m2 khai báo
def detect_title_area_mismatch(row) -> int:
    title = str(row.get("title", ""))
    area  = row.get("area_m2")
    if pd.isna(area):
        return 0
    # Tìm WxL trong title: "4x15", "5x20m", "4,5x18"
    m = re.search(r"(\d+[.,]?\d*)\s*[xX×]\s*(\d+[.,]?\d*)", title)
    if not m:
        return 0
    try:
        w = float(m.group(1).replace(",", "."))
        l = float(m.group(2).replace(",", "."))
        area_title = w * l
        # Mismatch nếu lệch > 20%
        # Ngưỡng 100%: chỉ flag khi WxL khác area_m2 gấp đôi (tránh false positive nở hậu, đất hình thang)
        return int(abs(area_title - float(area)) / float(area) > 1.00)
    except (ValueError, ZeroDivisionError):
        return 0

df["flag_v4_title_area_mismatch"] = df.apply(detect_title_area_mismatch, axis=1)

# Soft signals
CLICKBAIT_KW = [
    "vỡ nợ", "vo no", "cần tiền gấp", "can tien gap",
    "thanh lý", "thanh ly",
    "phát mại", "phat mai", "giải chấp", "giai chap",
    "ngộp", "ngop", "cắt lỗ", "cat lo",
    # "bán gấp" / "ban gap" bị loại — quá phổ biến trong tin hợp lệ,
    # không đủ tính phân biệt để dùng làm soft signal
]

def has_clickbait_low_price(row) -> int:
    text  = (str(row.get("title", "")) + " " + str(row.get("description", ""))).lower()
    rho_v = row.get("price_per_m2_calc")
    u_lo  = row.get("_u_lo")
    if pd.isna(rho_v) or pd.isna(u_lo):
        return 0
    has_kw = any(kw in text for kw in CLICKBAIT_KW)
    return int(has_kw and float(rho_v) < float(u_lo))

df["flag_v4_price_band_violation"] = ((r > THR_VIOLATION) & (r <= THR_SEVERE)).astype(int)
df["flag_v4_clickbait_low_price"]  = df.apply(has_clickbait_low_price, axis=1)
df["flag_v4_missing_basic"]        = (
    df["price_billion"].isna() | df["area_m2"].isna()
).astype(int)
df["flag_v4_missing_core_info"]    = (
    df["road_norm"].fillna("").eq("") | (df["word_count"].fillna(0) < 20)
).astype(int)

print(f"\n[6] Signal activation:")
for col in [
    "flag_v4_price_band_severe",
    "flag_v4_title_area_mismatch",
    "flag_v4_price_band_violation",
    "flag_v4_clickbait_low_price",
    "flag_v4_missing_basic",
    "flag_v4_missing_core_info",
]:
    n = int(df[col].sum())
    print(f"    {col:<42} {n:>5}  ({n/len(df)*100:.1f}%)")

# ══════════════════════════════════════════════════════════════════
# BƯỚC 7 — Gán nhãn
# ══════════════════════════════════════════════════════════════════

# Soft score
SOFT_WEIGHTS = {
    "flag_v4_price_band_violation":  3,
    "flag_v4_clickbait_low_price":   3,
    "flag_v4_missing_basic":         2,
    "flag_v4_missing_core_info":     2,
}
df["_v6_score"] = sum(df[c] * w for c, w in SOFT_WEIGHTS.items())

# Hard signal = severe OR mismatch
df["_v6_has_hard"] = (
    (df["flag_v4_price_band_severe"] == 1) |
    (df["flag_v4_title_area_mismatch"] == 1)
).astype(int)

# Label: hard signal HOẶC soft score ≥ ngưỡng
df["label_suspicious_v6_NEW"] = (
    (df["_v6_has_hard"] == 1) | (df["_v6_score"] >= THR_SOFT_MIN)
).astype(int)

n_sus   = int(df["label_suspicious_v6_NEW"].sum())
n_hard  = int(df["_v6_has_hard"].sum())
n_soft  = int(((df["_v6_has_hard"] == 0) & (df["label_suspicious_v6_NEW"] == 1)).sum())
sus_pct = n_sus / len(df) * 100

print(f"\n[7] Labeling results:")
print(f"    n total       = {len(df):,}")
print(f"    n suspicious  = {n_sus:,}  ({sus_pct:.2f}%)")
print(f"      └ hard only = {n_hard:,}")
print(f"      └ soft only = {n_soft:,}")

print(f"\n    Suspicious rate by property_cat:")
for cat, grp in df.groupby("_property_cat"):
    sr = grp["label_suspicious_v6_NEW"].mean() * 100
    print(f"      {cat:<20} {sr:>5.1f}%  (n={len(grp)})")

# ══════════════════════════════════════════════════════════════════
# BƯỚC 8 — Duplicate group signals
# ══════════════════════════════════════════════════════════════════

sig_counts = df["description_signature"].value_counts()
df["duplicate_group_size"] = (
    df["description_signature"].map(sig_counts).fillna(1).astype(int)
)

df["duplicate_phone_conflict"]    = 0
df["duplicate_location_conflict"] = 0

for sig, grp in df.groupby("description_signature"):
    if len(grp) <= 1:
        continue
    phones = grp["phone_numbers"].dropna().unique()
    dists  = grp["district_norm"].dropna().unique()
    df.loc[grp.index, "duplicate_phone_conflict"]    = int(len(phones) > 1)
    df.loc[grp.index, "duplicate_location_conflict"] = int(len(dists) > 1)

df["flag_duplicate_conflict"] = (
    (df["duplicate_phone_conflict"] == 1) |
    (df["duplicate_location_conflict"] == 1)
).astype(int)

df["duplicate_road_nunique"]     = df.groupby("description_signature")["road_norm"].transform("nunique")
df["duplicate_district_nunique"] = df.groupby("description_signature")["district_norm"].transform("nunique")
df["duplicate_province_nunique"] = df.groupby("description_signature")["province_norm"].transform("nunique")
df["duplicate_phone_nunique"]    = df.groupby("description_signature")["phone_numbers"].transform("nunique")

dup_gt1 = (df["duplicate_group_size"] > 1).sum()
print(f"\n[8] Duplicate signals: {dup_gt1} rows in groups > 1")

# ══════════════════════════════════════════════════════════════════
# BƯỚC 9 — Canonical label
# ══════════════════════════════════════════════════════════════════

df["_canonical_label"] = df["label_suspicious_v6_NEW"]

# ══════════════════════════════════════════════════════════════════
# BƯỚC 10 — Lưu output đầy đủ
# ══════════════════════════════════════════════════════════════════

df.to_csv(OUTPUT_FULL, index=False, encoding="utf-8-sig")
print(f"\n[9] Saved full output → {OUTPUT_FULL}  ({len(df)} rows × {len(df.columns)} cols)")

# ══════════════════════════════════════════════════════════════════
# BƯỚC 11 — Chọn cột cho STEP2 (model training)
#
# Giữ đúng 27 tabular features + 2 text features + label
# Loại bỏ:
#   - Các cột raw text gốc (title, description, location...)
#   - Các cột all-null trong dataset này
#   - Các cột nội bộ pipeline (_u_hi, _u_lo, _r_typeaware...)
#   - Các cột chỉ dùng để debug (v4_*, label_v4_*)
# ══════════════════════════════════════════════════════════════════

# 27 tabular features cho M2 (khớp đúng Table 6 bài báo)
FEAT_NUM = [
    "price_billion",
    "area_m2",
    "price_per_m2_calc",
    "bedrooms_num",
    "bathrooms_num",
    "usable_area_num",
    "length_num",
    "width_num",
    "land_area_num",
    "word_count",
    "digit_count",
    "exclamation_count",
    "text_length",
    "phone_like_count",
    "is_agri_land",
    "is_can_ho_premium",
    "has_extreme_clickbait",
    "clickbait_keyword_count",
    "flag_v4_parse_error",
    "flag_v4_title_area_mismatch",
    "flag_v4_missing_core_info",
    "duplicate_group_size",
    "duplicate_road_nunique",
    "duplicate_district_nunique",
    "duplicate_phone_nunique",
]

FEAT_CAT = [
    "district_norm",
    "province_norm",
    "frontage_class",
    "house_type_norm",
    "land_type_norm",
    "legal_documents_norm",
]

# Text features cho M1/M3/M4/M5/M6/M7
TEXT_COLS = [
    "full_text_no_price",   # TF-IDF input
    "full_text_norm",       # PhoBERT input
]

# Metadata (giữ để trace lại dễ hơn)
META_COLS = [
    "title",
    "location",
    "district",
    "province",
    "district_norm",
    "province_norm",
    "description_signature",
]

# Label
LABEL_COLS = [
    "label_suspicious_v6_NEW",
    "_canonical_label",
    "_property_cat",
    "_v6_has_hard",
    "_v6_score",
    "_threshold_source",
]

# Signal flags (hữu ích cho analysis, loại trước khi train)
FLAG_COLS = [
    "flag_v4_price_band_severe",
    "flag_v4_price_band_violation",
    "flag_v4_clickbait_low_price",
    "flag_v4_title_area_mismatch",
    "flag_v4_missing_basic",
    "flag_v4_missing_core_info",
    "flag_duplicate_conflict",
    "duplicate_phone_conflict",
    "duplicate_location_conflict",
]

# Tổng hợp các cột cần giữ
keep_cols = list(dict.fromkeys(
    META_COLS + FEAT_NUM + FEAT_CAT + TEXT_COLS + FLAG_COLS + LABEL_COLS
))

# Chỉ giữ cột thực sự tồn tại trong df
keep_cols = [c for c in keep_cols if c in df.columns]

# Thêm is_can_ho_premium, has_extreme_clickbait, clickbait_keyword_count
# nếu chưa có trong df (scraper v2 chưa tính)
for col in ["has_extreme_clickbait", "clickbait_keyword_count", "flag_v4_parse_error"]:
    if col not in df.columns:
        df[col] = 0

# Re-check sau khi fill
keep_cols = [c for c in keep_cols if c in df.columns]

df_model = df[keep_cols].copy()
df_model.to_csv(OUTPUT_MODEL, index=False, encoding="utf-8-sig")

print(f"\n[10] Model input saved → {OUTPUT_MODEL}")
print(f"     {len(df_model)} rows × {len(df_model.columns)} cols")
print(f"\n     Cột được giữ:")
print(f"       Metadata:       {len(META_COLS)} cols")
print(f"       Numeric feat:   {len([c for c in FEAT_NUM if c in df.columns])} cols")
print(f"       Categorical:    {len([c for c in FEAT_CAT if c in df.columns])} cols")
print(f"       Text feat:      {len([c for c in TEXT_COLS if c in df.columns])} cols")
print(f"       Signal flags:   {len([c for c in FLAG_COLS if c in df.columns])} cols")
print(f"       Labels:         {len([c for c in LABEL_COLS if c in df.columns])} cols")

print(f"\n     Cột bị loại bỏ ({len(df.columns) - len(df_model.columns)}):")
dropped = [c for c in df.columns if c not in keep_cols]
for c in dropped:
    print(f"       - {c}")

print("\n" + "=" * 60)
print("STEP1 hoàn thành.")
print(f"  → Gửi file '{OUTPUT_MODEL}' vào STEP2 để train model.")
print("=" * 60)

---
## STEP 2 — Model Training & Evaluation

Huấn luyện và đánh giá 7 cấu hình mô hình bằng Stratified 5-fold CV.

**Input:** `mogi_model_input.csv`  
**Output:** `step2_mogi_output.txt` · `roc_data_mogi.csv` · `feature_importance_mogi.csv`

Chạy không có PhoBERT (nhanh hơn): thêm `--skip-phobert` vào `sys.argv` trước khi chạy cell này.

In [ ]:
import sys, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score, roc_curve, confusion_matrix
)
from sklearn.inspection import permutation_importance

# ── CONFIG ────────────────────────────────────────────────────
DATA_PATH      = 'mogi_model_input.csv'
LABEL_COL      = 'label_suspicious_v6_NEW'
TEXT_COL       = 'full_text_no_price'
N_SPLITS       = 5
RANDOM_STATE   = 42
SKIP_PHOBERT   = '--skip-phobert' in sys.argv
PHOBERT_MODEL  = 'vinai/phobert-base'
PHOBERT_MAX_LEN = 128

# 22 numeric features (loại has_extreme_clickbait / clickbait_keyword_count
# / flag_v4_parse_error vì không có trong mogi dataset)
# 20 numeric features
# Ghi chú: flag_v4_title_area_mismatch và flag_v4_missing_core_info
# bị loại vì là thành phần của công thức gán nhãn STEP1
# (flag_v4_title_area_mismatch là hard signal trực tiếp tạo label,
#  flag_v4_missing_core_info là soft signal trọng số 2)
# → đưa vào FEAT_NUM = label leak. Paper gốc không dùng flag_v4_* nào.
FEAT_NUM = [
    'price_billion', 'area_m2', 'price_per_m2_calc',
    'bedrooms_num', 'bathrooms_num', 'usable_area_num',
    'length_num', 'width_num', 'land_area_num',
    'word_count', 'digit_count', 'exclamation_count',
    'text_length', 'phone_like_count',
    'is_agri_land', 'is_can_ho_premium',
    'duplicate_group_size', 'duplicate_road_nunique',
    'duplicate_district_nunique', 'duplicate_phone_nunique',
]

# 6 categorical features (giống bài báo gốc)
FEAT_CAT = [
    'frontage_class', 'district_norm', 'province_norm',
    'house_type_norm', 'land_type_norm', 'legal_documents_norm',
]

# ── HELPERS ───────────────────────────────────────────────────

def load_data(path):
    df = pd.read_csv(path, low_memory=False)
    df[LABEL_COL] = df[LABEL_COL].fillna(0).astype(int)
    for c in FEAT_NUM:
        df[c] = pd.to_numeric(df.get(c, 0), errors='coerce').fillna(0)
    for c in FEAT_CAT:
        df[c] = df[c].fillna('unknown').astype(str) if c in df.columns else 'unknown'
    if TEXT_COL not in df.columns:
        df[TEXT_COL] = ''
    df[TEXT_COL] = df[TEXT_COL].fillna('')
    return df

def build_tabular(df):
    num_arr   = df[[c for c in FEAT_NUM if c in df.columns]].values.astype(float)
    cat_parts = []
    for c in FEAT_CAT:
        col = df[c].astype(str) if c in df.columns else pd.Series(['unknown']*len(df))
        le  = LabelEncoder()
        cat_parts.append(le.fit_transform(col).reshape(-1,1).astype(float))
    X_tab = np.hstack([num_arr] + cat_parts)
    feat_names = [c for c in FEAT_NUM if c in df.columns] + FEAT_CAT
    return X_tab, feat_names

def get_metrics(y_true, y_pred, y_prob):
    return {
        'prec': precision_score(y_true, y_pred, zero_division=0),
        'rec':  recall_score(y_true, y_pred, zero_division=0),
        'f1':   f1_score(y_true, y_pred, zero_division=0),
        'acc':  accuracy_score(y_true, y_pred),
        'auc':  roc_auc_score(y_true, y_prob),
    }

def run_cv(model_fn, X, y):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    fold_metrics = []
    all_preds = np.zeros(len(y), dtype=int)
    all_probs = np.zeros(len(y))
    for tr, te in skf.split(X, y):
        clf = model_fn()
        clf.fit(X[tr], y[tr])
        pred = clf.predict(X[te])
        prob = clf.predict_proba(X[te])[:, 1]
        all_preds[te] = pred
        all_probs[te] = prob
        fold_metrics.append(get_metrics(y[te], pred, prob))
    result = {}
    for k in ['prec', 'rec', 'f1', 'acc', 'auc']:
        vals = [m[k] for m in fold_metrics]
        result[f'{k}_mean'] = np.mean(vals)
        result[f'{k}_std']  = np.std(vals, ddof=1)
    result['all_preds'] = all_preds
    result['all_probs'] = all_probs
    return result

def mcnemar_test(y_true, preds_a, preds_b):
    ca = (preds_a == y_true); cb = (preds_b == y_true)
    b  = np.sum(ca & ~cb);    c  = np.sum(~ca & cb)
    if b + c == 0: return 1.0
    stat = (abs(b - c) - 1)**2 / (b + c)
    return 1 - stats.chi2.cdf(stat, df=1)

def bootstrap_ci(y_true, all_preds, all_probs, n_boot=1000, alpha=0.95):
    rng = np.random.default_rng(RANDOM_STATE)
    n   = len(y_true)
    f1s, aucs = [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt, yp, yprob = y_true[idx], all_preds[idx], all_probs[idx]
        if len(np.unique(yt)) < 2: continue
        f1s.append(f1_score(yt, yp, zero_division=0))
        aucs.append(roc_auc_score(yt, yprob))
    lo = (1 - alpha) / 2
    return {
        'f1_ci_lo':  np.quantile(f1s, lo),
        'f1_ci_hi':  np.quantile(f1s, 1-lo),
        'auc_ci_lo': np.quantile(aucs, lo),
        'auc_ci_hi': np.quantile(aucs, 1-lo),
    }

def roc_df(y_true, all_probs, name):
    fpr, tpr, _ = roc_curve(y_true, all_probs)
    idx = np.round(np.linspace(0, len(fpr)-1, 200)).astype(int)
    return pd.DataFrame({'model': name, 'fpr': fpr[idx], 'tpr': tpr[idx]})

# ── PHOBERT ───────────────────────────────────────────────────

def extract_phobert(texts, batch_size=32):
    try:
        from transformers import AutoTokenizer, AutoModel
        import torch
    except ImportError:
        print("  [LỖI] pip install transformers torch"); sys.exit(1)
    print(f"  Đang tải {PHOBERT_MODEL}...")
    tok = AutoTokenizer.from_pretrained(PHOBERT_MODEL)
    mdl = AutoModel.from_pretrained(PHOBERT_MODEL); mdl.eval()
    device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
    mdl.to(device)
    print(f"  Device: {device}")
    try:
        from underthesea import word_tokenize
        texts = [word_tokenize(t, format='text') for t in texts]
        print("  Tách từ: underthesea ✓")
    except ImportError:
        print("  [Cảnh báo] underthesea không có, dùng text thô.")
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tok(batch, padding=True, truncation=True,
                  max_length=PHOBERT_MAX_LEN, return_tensors='pt').to(device)
        with __import__('torch').no_grad():
            out = mdl(**enc)
        vecs.append(out.last_hidden_state[:, 0, :].cpu().numpy())
        print(f"  PhoBERT: {min(100, int(100*(i+batch_size)/len(texts)))}%...", end='\r')
    print()
    return np.vstack(vecs)

# ── MAIN ──────────────────────────────────────────────────────

def main():
    t0 = time.time()
    lines = []
    def pr(s=''):
        print(s); lines.append(str(s))

    pr('='*65)
    pr('SAD-VREAL — STEP2 MOGI (CV 5-fold, 7 mô hình)')
    pr('='*65)

    # 1. Load
    pr(f'\nĐọc {DATA_PATH}...')
    df = load_data(DATA_PATH)
    y  = df[LABEL_COL].values
    n_sus, n_tot = int(y.sum()), len(y)
    pr(f'  n_total = {n_tot:,}   n_sus = {n_sus:,}  ({100*n_sus/n_tot:.2f}%)')
    pr(f'  HCM: {(df["province_norm"]=="tp ho chi minh").sum():,}'
       f'  HN: {(df["province_norm"]=="ha noi").sum():,}')

    # 2. Build features
    pr('\nXây dựng feature matrices...')
    X_tab, feat_names_tab = build_tabular(df)
    pr(f'  X_tab: {X_tab.shape}  ({len([c for c in FEAT_NUM if c in df.columns])} num + {len(FEAT_CAT)} cat)')

    pr('  TF-IDF word-level...')
    tfidf_w   = TfidfVectorizer(ngram_range=(1,2), max_features=10000, sublinear_tf=True, min_df=2)
    X_tfidf_w = tfidf_w.fit_transform(df[TEXT_COL])
    X_svd     = TruncatedSVD(n_components=100, random_state=RANDOM_STATE).fit_transform(X_tfidf_w)

    pr('  TF-IDF char-level...')
    tfidf_c   = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), max_features=8000, sublinear_tf=True, min_df=3)
    X_tfidf_c = tfidf_c.fit_transform(df[TEXT_COL])
    X_svd_c   = TruncatedSVD(n_components=80, random_state=RANDOM_STATE).fit_transform(X_tfidf_c)

    feats = {
        'y':     y,
        'X_tab': X_tab,
        'X_m1':  X_tfidf_w.toarray().astype(np.float32),
        'X_m3':  np.hstack([X_svd, X_tab]),
        'X_m4':  np.hstack([X_tfidf_w.toarray(), X_tfidf_c.toarray()]).astype(np.float32),
        'X_m5':  np.hstack([X_svd, X_svd_c, X_tab]),
        'X_svd': X_svd,
        'feat_names_tab': feat_names_tab,
    }

    # 3. PhoBERT
    if not SKIP_PHOBERT:
        pr('\nTrích đặc trưng PhoBERT M6/M7...')
        X_pb = extract_phobert(df[TEXT_COL].tolist())
        feats['X_m6'] = X_pb
        feats['X_m7'] = np.hstack([X_pb, X_tab])
        pr(f'  PhoBERT: {X_pb.shape}')
    else:
        pr('\n[Bỏ qua M6/M7 theo --skip-phobert]')

    # 4. Model factories
    def lr():  return LogisticRegression(max_iter=1000, C=1.0, class_weight='balanced',
                                          random_state=RANDOM_STATE, n_jobs=-1)
    def hgb(): return HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05,
                        max_leaf_nodes=31, min_samples_leaf=20,
                        class_weight='balanced', random_state=RANDOM_STATE)

    models = {
        'M1 Văn bản (từ)':           ('X_m1',  lr),
        'M2 Bảng (HGB)':             ('X_tab', hgb),
        'M3 Đa phương thức':         ('X_m3',  hgb),
        'M4 Văn bản (từ+ký tự)':     ('X_m4',  lr),
        'M5 Đa phương thức đầy đủ':  ('X_m5',  hgb),
    }
    if not SKIP_PHOBERT and 'X_m6' in feats:
        models['M6 PhoBERT (văn bản)']    = ('X_m6', lr)
        models['M7 PhoBERT + bảng (HGB)'] = ('X_m7', hgb)

    # 5. CV
    pr('\n' + '-'*65)
    pr('BẢNG KẾT QUẢ — CV 5-fold')
    pr('-'*65)
    pr(f"{'Mô hình':<35} {'Prec':>6} {'Rec':>6} {'F1':>6} {'Acc':>6} {'AUC':>6}")
    pr(f"{'':35} {'±std':>6} {'±std':>6} {'±std':>6} {'±std':>6} {'±std':>6}")
    pr('-'*65)

    cv_results = {}; roc_dfs = []
    for name, (xkey, mfn) in models.items():
        t1 = time.time()
        print(f'  {name}...', end=' ', flush=True)
        res = run_cv(mfn, feats[xkey], y)
        ci  = bootstrap_ci(y, res['all_preds'], res['all_probs'])
        res.update(ci); cv_results[name] = res
        elapsed = time.time() - t1
        pr(f"{name:<35} {res['prec_mean']:>6.3f} {res['rec_mean']:>6.3f} "
           f"{res['f1_mean']:>6.3f} {res['acc_mean']:>6.3f} {res['auc_mean']:>6.3f}")
        pr(f"{'':35} {res['prec_std']:>6.3f} {res['rec_std']:>6.3f} "
           f"{res['f1_std']:>6.3f} {res['acc_std']:>6.3f} {res['auc_std']:>6.3f}")
        print(f'({elapsed:.0f}s)')
        roc_dfs.append(roc_df(y, res['all_probs'], name))

    # 6. Bootstrap CI
    pr('\n' + '-'*65)
    pr('BOOTSTRAP 95% CI — F1 và AUC  (n_boot=1000)')
    pr('-'*65)
    pr(f"{'Mô hình':<35} {'F1 [lo–hi]':>20} {'AUC [lo–hi]':>20}")
    pr('-'*65)
    for name, res in cv_results.items():
        pr(f"{name:<35} [{res['f1_ci_lo']:.3f} – {res['f1_ci_hi']:.3f}]"
           f"   [{res['auc_ci_lo']:.3f} – {res['auc_ci_hi']:.3f}]")

    # 7. McNemar — M2 vs tất cả
    pr('\n' + '-'*65)
    pr('MCNEMAR TEST — M2 vs. các cấu hình')
    pr('-'*65)
    if 'M2 Bảng (HGB)' in cv_results:
        m2_preds = cv_results['M2 Bảng (HGB)']['all_preds']
        pr(f"{'So sánh':<45} {'p-value':>10}  Kết luận"); pr('-'*65)
        for name, res in cv_results.items():
            if name == 'M2 Bảng (HGB)': continue
            p = mcnemar_test(y, m2_preds, res['all_preds'])
            conclude = 'Khác biệt đáng kể (p<0.05)' if p < 0.05 else 'Không đáng kể'
            pr(f"{'M2 vs ' + name:<45} {p:>10.4f}  {conclude}")

    # 8. Ablation M3
    pr('\n' + '-'*65)
    pr('ABLATION — Đóng góp phương thức (M3, CV 5-fold)')
    pr('-'*65)
    n_num = len([c for c in FEAT_NUM if c in df.columns])
    ablation = {
        'Đầy đủ (văn bản + bảng)':                  feats['X_m3'],
        'Chỉ bảng (bỏ văn bản)':                    feats['X_tab'],
        'Chỉ văn bản (bỏ bảng)':                    feats['X_svd'],
        f'Non-categorical tabular ({n_num} features)': feats['X_tab'][:, :n_num],
    }
    pr(f"{'Cấu hình':<42} {'Prec':>6} {'Rec':>6} {'F1':>6} {'Acc':>6}"); pr('-'*65)
    for cfg, X_cfg in ablation.items():
        r = run_cv(hgb, X_cfg, y)
        pr(f"{cfg:<42} {r['prec_mean']:>6.3f} {r['rec_mean']:>6.3f} "
           f"{r['f1_mean']:>6.3f} {r['acc_mean']:>6.3f}")

    # 9. Permutation importance M3
    pr('\n' + '-'*65)
    pr('FEATURE IMPORTANCE — Permutation (M3, fold 1, n_repeats=10)')
    pr('-'*65)
    skf_fi = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    tr_i, te_i = next(skf_fi.split(feats['X_m3'], y))
    clf_fi = hgb(); clf_fi.fit(feats['X_m3'][tr_i], y[tr_i])
    pi = permutation_importance(clf_fi, feats['X_m3'][te_i], y[te_i],
                                n_repeats=10, random_state=RANDOM_STATE,
                                scoring='f1', n_jobs=-1)
    fi_names = [f'tfidf_svd_{i}' for i in range(100)] + feats['feat_names_tab']
    fi_df = (pd.DataFrame({'feature': fi_names,
                            'importance': pi.importances_mean,
                            'std': pi.importances_std})
               .sort_values('importance', ascending=False).head(20))
    pr(f"{'Rank':<5} {'Feature':<35} {'Importance':>12} {'±Std':>8}"); pr('-'*65)
    fi_records = []
    for rank, (_, row) in enumerate(fi_df.iterrows(), 1):
        pr(f"{rank:<5} {row['feature']:<35} {row['importance']:>12.4f} {row['std']:>8.4f}")
        fi_records.append({'rank': rank, 'feature': row['feature'],
                           'importance': row['importance'], 'std': row['std']})

    # 10. Per-segment M3
    pr('\n' + '-'*65)
    pr('PHÂN TÍCH THEO PHÂN KHÚC GIÁ — M3')
    pr('-'*65)
    m3_preds = cv_results['M3 Đa phương thức']['all_preds']
    m3_probs = cv_results['M3 Đa phương thức']['all_probs']
    bins   = [0, 1, 3, 10, 50, 1e9]
    labels = ['<1 tỷ', '1–3 tỷ', '3–10 tỷ', '10–50 tỷ', '>50 tỷ']
    df['price_seg'] = pd.cut(df['price_billion'], bins=bins, labels=labels)
    pr(f"{'Phân khúc':<12} {'n':>6} {'%sus':>8} {'F1':>8} {'Prec':>8} {'Rec':>8}"); pr('-'*65)
    for seg in labels:
        mask = (df['price_seg'] == seg).values
        if mask.sum() == 0: continue
        yt, yp = y[mask], m3_preds[mask]
        f1   = f1_score(yt, yp, zero_division=0)
        prec = precision_score(yt, yp, zero_division=0)
        rec  = recall_score(yt, yp, zero_division=0)
        pr(f"{seg:<12} {mask.sum():>6} {100*yt.mean():>7.1f}% "
           f"{f1:>8.3f} {prec:>8.3f} {rec:>8.3f}")

    # 11. HCM vs HN M3
    pr('\n' + '-'*65)
    pr('PHÂN TÍCH THEO ĐỊA LÝ — M3 (HCM vs HN)')
    pr('-'*65)
    pr(f"{'Vùng':<20} {'n':>6} {'%sus':>8} {'F1':>8} {'Prec':>8} {'Rec':>8}"); pr('-'*65)
    for prov_key, prov_label in [('tp ho chi minh','TP.HCM'), ('ha noi','Hà Nội')]:
        mask = (df['province_norm'] == prov_key).values
        if mask.sum() == 0: continue
        yt, yp = y[mask], m3_preds[mask]
        f1   = f1_score(yt, yp, zero_division=0)
        prec = precision_score(yt, yp, zero_division=0)
        rec  = recall_score(yt, yp, zero_division=0)
        pr(f"{prov_label:<20} {mask.sum():>6} {100*yt.mean():>7.1f}% "
           f"{f1:>8.3f} {prec:>8.3f} {rec:>8.3f}")

    # 12. Confusion matrix M3
    pr('\n' + '-'*65)
    pr('CONFUSION MATRIX — M3 (aggregated OOF)')
    pr('-'*65)
    cm = confusion_matrix(y, m3_preds)
    tn, fp, fn, tp_ = cm.ravel()
    pr(f'  TN={tn}  FP={fp}  FN={fn}  TP={tp_}')
    pr(f'  FPR = {fp/(tn+fp):.3f}   FNR = {fn/(fn+tp_):.3f}')

    # Summary
    pr('\n' + '='*65)
    pr(f'TỔNG THỜI GIAN: {time.time()-t0:.0f} giây')
    pr('='*65)

    # Save
    with open('step2_mogi_output.txt', 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    pd.concat(roc_dfs, ignore_index=True).to_csv('roc_data_mogi.csv', index=False)
    pd.DataFrame(fi_records).to_csv('feature_importance_mogi.csv', index=False)

    print('\nĐã lưu:')
    print('  step2_mogi_output.txt')
    print('  roc_data_mogi.csv')
    print('  feature_importance_mogi.csv')
    print('\nGửi 3 file này cho Claude để xử lý tiếp.')

if __name__ == '__main__':
    main()

SAD-VREAL — STEP2 MOGI (CV 5-fold, 7 mô hình)

Đọc mogi_model_input.csv...
  n_total = 4,866   n_sus = 501  (10.30%)
  HCM: 2,735  HN: 2,131

Xây dựng feature matrices...
  X_tab: (4866, 26)  (20 num + 6 cat)
  TF-IDF word-level...
  TF-IDF char-level...

Trích đặc trưng PhoBERT M6/M7...
  Đang tải vinai/phobert-base...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 30726.94it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Device: cpu


---
## STEP 3 — Figure Generation

Tạo 8 hình minh họa dùng trong bài báo.

**Input:** `mogi_labeled.csv` · `roc_data_mogi.csv` · `feature_importance_mogi.csv`  
**Output:** thư mục `figures/` — mỗi figure có cả `.png` và `.pdf`

| File | Nội dung |
|------|----------|
| `fig_preprocess.png` | Pipeline tiền xử lý 4,965 → 4,866 |
| `fig_flags.png` | Tần suất 6 tín hiệu gắn nhãn |
| `fig_cv_all7.png` | So sánh F₁ 7 mô hình |
| `fig_roc.png` | ROC curves |
| `fig_confmat.png` | Confusion matrix M3 |
| `fig_ablation.png` | Ablation study |
| `fig_featimp.png` | Permutation importance top-12 |
| `fig_price_seg.png` | Phân tích theo phân khúc giá |

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap

# ── Paths ──────────────────────────────────────────────────────
LABELED_CSV  = "mogi_labeled.csv"
ROC_CSV      = "roc_data_mogi.csv"
FEAT_CSV     = "feature_importance_mogi.csv"
OUT_DIR      = "figures"
os.makedirs(OUT_DIR, exist_ok=True)

# ── Style constants ────────────────────────────────────────────
C_NAVY   = "#1C3F6E"
C_ORANGE = "#F4831F"
C_GRAY   = "#9CA3AF"
C_RED    = "#DC2626"
C_TEAL   = "#0E7490"
C_GRID   = "#E2E8F0"
C_TEXT   = "#1E293B"

def save(fig, name):
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, dpi=180, bbox_inches="tight", facecolor="white")
    fig.savefig(path.replace(".png", ".pdf"), bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Saved: {name}")

# ══════════════════════════════════════════════════════════════
# Load data once
# ══════════════════════════════════════════════════════════════
print("Loading data...")
df      = pd.read_csv(LABELED_CSV, low_memory=False)
roc_df  = pd.read_csv(ROC_CSV)
feat_df = pd.read_csv(FEAT_CSV)

LBL = "label_suspicious_v6_NEW"
n_total = len(df)
n_sus   = int(df[LBL].sum())
print(f"  n={n_total}, sus={n_sus} ({n_sus/n_total*100:.2f}%)")

# ══════════════════════════════════════════════════════════════
# FIG 1 — Preprocessing pipeline
# ══════════════════════════════════════════════════════════════
print("\nFig 1: fig_preprocess")

steps  = [4965, 4965, 4915, 4880, 4866]
labels = ["HCM+HN ads\n(initial)",
          "After price\nnorm. (0 removed)",
          "After exact\ndedup (−50)",
          "After near-\ndedup (−35)",
          "After price/m²\nfilter (−14)"]
colors = [C_GRAY, C_GRAY, C_NAVY, C_NAVY, C_ORANGE]

fig, ax = plt.subplots(figsize=(9, 3.6))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

bar_w = 0.55
x = np.arange(len(steps))
bars = ax.bar(x, steps, width=bar_w, color=colors, linewidth=0)

# Value labels on top
for xi, val in zip(x, steps):
    ax.text(xi, val + 18, f"{val:,}", ha="center", va="bottom",
            fontsize=12, fontweight="bold", color=C_TEXT)

# Arrows between bars
for i in range(len(steps) - 1):
    ax.annotate("",
        xy=(x[i+1] - bar_w/2 - 0.05, steps[i+1] * 0.5),
        xytext=(x[i] + bar_w/2 + 0.05, steps[i] * 0.5),
        arrowprops=dict(arrowstyle="->", color=C_GRAY, lw=1.5))

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylim(0, max(steps) * 1.18)
ax.set_title("Data preprocessing pipeline (n=4,866)", fontsize=12, fontweight="bold", pad=8)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.yaxis.set_visible(False)
ax.set_axisbelow(True)

plt.tight_layout(pad=0.8)
save(fig, "fig_preprocess.png")

# ══════════════════════════════════════════════════════════════
# FIG 2 — Activation frequency of 6 signals
# ══════════════════════════════════════════════════════════════
print("Fig 2: fig_flags")

# Compute from labeled CSV
flag_cols = {
    "price_band_severe":   "flag_v4_price_band_severe",
    "price_band_violation":"flag_v4_price_band_violation",
    "missing_core_info":   "flag_v4_missing_core_info",
    "clickbait_low_price": "flag_v4_clickbait_low_price",
    "title_area_mismatch": "flag_v4_title_area_mismatch",
    "missing_basic":       "flag_v4_missing_basic",
}
is_hard = {
    "price_band_severe": True,
    "title_area_mismatch": True,
    "price_band_violation": False,
    "clickbait_low_price": False,
    "missing_basic": False,
    "missing_core_info": False,
}
sig_names  = list(flag_cols.keys())
sig_counts = [int(df[flag_cols[s]].sum()) for s in sig_names]
sig_colors = [C_ORANGE if is_hard[s] else C_NAVY for s in sig_names]

fig, ax = plt.subplots(figsize=(8, 4.2))
fig.patch.set_facecolor("white"); ax.set_facecolor("white")

y_pos = np.arange(len(sig_names))
ax.barh(y_pos, sig_counts, color=sig_colors, height=0.55, linewidth=0)

for i, val in enumerate(sig_counts):
    xoff = val + 5 if val > 0 else 5
    ax.text(xoff, y_pos[i], str(val), va="center", ha="left",
            fontsize=11, color=C_TEXT, fontweight="bold")

ax.set_yticks(y_pos)
ax.set_yticklabels(sig_names, fontsize=11)
ax.set_xlabel("Number of activated ads", fontsize=11)
ax.set_xlim(0, max(sig_counts) * 1.18)
ax.set_title(f"Activation frequency of the six labeling signals\n(n={n_total:,})",
             fontsize=12, fontweight="bold", pad=10)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.xaxis.grid(True, color=C_GRID, linewidth=0.6)
ax.set_axisbelow(True)

hard_p = mpatches.Patch(color=C_ORANGE, label="Hard signals")
soft_p = mpatches.Patch(color=C_NAVY,   label="Soft signals")
ax.legend(handles=[hard_p, soft_p], loc="upper right",
          frameon=True, framealpha=0.95, fontsize=10,
          handlelength=1.2, borderpad=0.6, labelspacing=0.4)

plt.tight_layout(pad=0.8)
save(fig, "fig_flags.png")

# ══════════════════════════════════════════════════════════════
# FIG 3 — F1 comparison all 7 models
# ══════════════════════════════════════════════════════════════
print("Fig 3: fig_cv_all7")

model_labels = ["M1", "M2", "M3", "M4", "M5", "M6", "M7"]
f1_means = [0.469, 0.831, 0.794, 0.480, 0.787, 0.332, 0.745]
f1_stds  = [0.020, 0.015, 0.012, 0.017, 0.025, 0.025, 0.011]
bar_colors = [C_ORANGE if i == 1 else
              C_TEAL   if i in (2, 4) else
              "#8B5CF6" if i == 6 else
              "#94A3B8"
              for i in range(7)]

fig, ax = plt.subplots(figsize=(8, 4.5))
fig.patch.set_facecolor("white"); ax.set_facecolor("white")

x = np.arange(len(model_labels))
bars = ax.bar(x, f1_means, yerr=f1_stds, width=0.6, color=bar_colors,
              capsize=4, error_kw=dict(elinewidth=1.2, ecolor="#475569"), linewidth=0)

# Best model dashed line
ax.axhline(max(f1_means), color=C_ORANGE, linewidth=1.4,
           linestyle="--", alpha=0.7, zorder=0)

for xi, (val, std) in enumerate(zip(f1_means, f1_stds)):
    weight = "bold" if xi == 1 else "normal"
    ax.text(xi, val + std + 0.018, f"{val:.3f}",
            ha="center", va="bottom", fontsize=10,
            fontweight=weight, color=C_TEXT)

ax.set_xticks(x)
ax.set_xticklabels(model_labels, fontsize=11)
ax.set_ylabel(r"$F_1$", fontsize=12)
ax.set_ylim(0, 1.0)
ax.set_title(fr"$F_1$ comparison — 7 models, 5-fold CV (n={n_total:,})",
             fontsize=12, fontweight="bold", pad=8)
ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.grid(True, color=C_GRID, linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout(pad=0.8)
save(fig, "fig_cv_all7.png")

# ══════════════════════════════════════════════════════════════
# FIG 4 — ROC curves
# ══════════════════════════════════════════════════════════════
print("Fig 4: fig_roc")

# Map model names to AUC + style
model_style = {
    "M1 Văn bản (từ)":          dict(color="#94A3B8", ls="-",    lw=1.5, auc=0.862),
    "M2 Bảng (HGB)":            dict(color=C_ORANGE,  ls="-",    lw=2.5, auc=0.980),
    "M3 Đa phương thức":        dict(color=C_NAVY,    ls="-",    lw=2.0, auc=0.974),
    "M4 Văn bản (từ+ký tự)":   dict(color="#CBD5E1", ls="--",   lw=1.5, auc=0.858),
    "M5 Đa phương thức đầy đủ": dict(color=C_TEAL,   ls="--",   lw=1.8, auc=0.972),
    "M6 PhoBERT (văn bản)":     dict(color=C_RED,    ls=":",    lw=1.5, auc=0.734),
    "M7 PhoBERT + bảng (HGB)":  dict(color="#8B5CF6", ls="-.",  lw=1.8, auc=0.962),
}

fig, ax = plt.subplots(figsize=(6.5, 5.5))
fig.patch.set_facecolor("white"); ax.set_facecolor("white")

ax.plot([0, 1], [0, 1], color=C_GRAY, lw=1, ls="--", alpha=0.6)  # diagonal

for model_name, style in model_style.items():
    sub = roc_df[roc_df["model"] == model_name].sort_values("fpr")
    if sub.empty:
        print(f"    WARNING: no ROC data for '{model_name}'")
        continue
    # Short label
    short = model_name.split(" ")[0] + " " + model_name.split(" ")[1]
    ax.plot(sub["fpr"], sub["tpr"],
            color=style["color"], ls=style["ls"], lw=style["lw"],
            label=f"{short}, AUC={style['auc']:.3f}")

ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title(f"Empirical ROC — CV 5-fold (n={n_total:,})",
             fontsize=12, fontweight="bold", pad=8)
ax.legend(fontsize=8.5, loc="lower right", frameon=True, framealpha=0.9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.spines[["top", "right"]].set_visible(False)
ax.xaxis.grid(True, color=C_GRID, linewidth=0.4)
ax.yaxis.grid(True, color=C_GRID, linewidth=0.4)

plt.tight_layout(pad=0.8)
save(fig, "fig_roc.png")

# ══════════════════════════════════════════════════════════════
# FIG 5 — Confusion matrix M3
# ══════════════════════════════════════════════════════════════
print("Fig 5: fig_confmat")

TN, FP, FN, TP = 8096, 118, 245, 701
total_cm = TN + FP + FN + TP
FPR_cm   = FP / (FP + TN)
FNR_cm   = FN / (FN + TP)

cm_vals  = [[TN, FP], [FN, TP]]
cm_pcts  = [[TN/(TN+FP), FP/(TN+FP)], [FN/(FN+TP), TP/(FN+TP)]]
cell_lbls= [["TN", "FP"], ["FN", "TP"]]
row_lbls = ["Normal\n(actual)", "Suspicious\n(actual)"]
col_lbls = ["Normal\n(predicted)", "Suspicious\n(predicted)"]

# Color: TN+TP dark, FN+FP light
bg_colors = [[C_NAVY,        "#BFDBFE"],
             ["#BFDBFE",     C_TEAL]]
fg_colors = [["white",       C_NAVY],
             [C_NAVY,        "white"]]

fig, ax = plt.subplots(figsize=(5.0, 4.2))
fig.patch.set_facecolor("white"); ax.set_facecolor("white")
ax.set_xlim(0, 2); ax.set_ylim(0, 2)
ax.set_aspect("equal")
ax.axis("off")

for r in range(2):
    for c in range(2):
        rect = plt.Rectangle([c, 1-r], 1, 1,
                              facecolor=bg_colors[r][c], edgecolor="white", lw=3)
        ax.add_patch(rect)
        val  = cm_vals[r][c]
        pct  = cm_pcts[r][c] * 100
        lbl  = cell_lbls[r][c]
        fg   = fg_colors[r][c]
        ax.text(c + 0.5, 1 - r + 0.58, f"{lbl}={val:,}",
                ha="center", va="center", fontsize=13,
                fontweight="bold", color=fg)
        ax.text(c + 0.5, 1 - r + 0.35, f"({pct:.1f}%)",
                ha="center", va="center", fontsize=11, color=fg)

# Axis labels
for i, lbl in enumerate(col_lbls):
    ax.text(i + 0.5, -0.08, lbl, ha="center", va="top", fontsize=9.5, color="#475569")
for i, lbl in enumerate(row_lbls):
    ax.text(-0.08, 1 - i + 0.5, lbl, ha="right", va="center", fontsize=9.5, color="#475569")

ax.set_title(f"M3 confusion matrix (CV 5-fold, n={total_cm:,})\n"
             f"FPR={FPR_cm:.3f}; FNR={FNR_cm:.3f}",
             fontsize=11, fontweight="bold", pad=10)

plt.tight_layout(pad=0.6)
save(fig, "fig_confmat.png")

# ══════════════════════════════════════════════════════════════
# FIG 6 — Ablation study
# ══════════════════════════════════════════════════════════════
print("Fig 6: fig_ablation")

abl_configs = ["Full\n(text+tab)", "Tabular\nonly", "Text\nonly", "Non-cat.\ntabular\n(20 feat)"]
abl_prec    = [0.856, 0.841, 0.538, 0.584]
abl_rec     = [0.741, 0.822, 0.336, 0.673]
abl_f1      = [0.794, 0.831, 0.413, 0.625]

x = np.arange(len(abl_configs))
w = 0.24

fig, ax = plt.subplots(figsize=(7.5, 4.5))
fig.patch.set_facecolor("white"); ax.set_facecolor("white")

b1 = ax.bar(x - w,     abl_prec, w, label="Precision", color=C_NAVY,   linewidth=0)
b2 = ax.bar(x,         abl_rec,  w, label="Recall",    color=C_ORANGE, linewidth=0)
b3 = ax.bar(x + w,     abl_f1,   w, label=r"$F_1$",   color=C_TEAL,   linewidth=0)

ax.set_xticks(x)
ax.set_xticklabels(abl_configs, fontsize=10)
ax.set_ylabel("Score", fontsize=11)
ax.set_ylim(0, 1.0)
ax.set_title(f"Quantitative ablation — modality contribution\n(CV 5-fold, n={n_total:,})",
             fontsize=12, fontweight="bold", pad=8)
ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.grid(True, color=C_GRID, linewidth=0.5)
ax.set_axisbelow(True)
ax.legend(fontsize=10, frameon=True, loc="upper right")

plt.tight_layout(pad=0.8)
save(fig, "fig_ablation.png")

# ══════════════════════════════════════════════════════════════
# FIG 7 — Feature importance
# ══════════════════════════════════════════════════════════════
print("Fig 7: fig_featimp")

top12 = feat_df.head(12).copy()
top12 = top12.sort_values("importance")  # ascending for horizontal bar

def feat_color(name):
    if name == "price_per_m2_calc":
        return C_ORANGE
    elif name in ("district_norm", "province_norm", "house_type_norm", "frontage_class"):
        return C_NAVY
    else:
        return C_TEAL

colors_fi = [feat_color(n) for n in top12["feature"]]

fig, ax = plt.subplots(figsize=(7.5, 5.0))
fig.patch.set_facecolor("white"); ax.set_facecolor("white")

y_pos = np.arange(len(top12))
ax.barh(y_pos, top12["importance"], xerr=top12["std"],
        color=colors_fi, height=0.55, linewidth=0,
        error_kw=dict(elinewidth=1.0, ecolor="#475569", capsize=3))

ax.set_yticks(y_pos)
ax.set_yticklabels(top12["feature"], fontsize=10)
ax.set_xlabel(r"Permutation importance ($\Delta F_1$)", fontsize=11)
ax.set_title(f"Top 12 features — permutation importance\n(M3, n={n_total:,})",
             fontsize=12, fontweight="bold", pad=8)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.xaxis.grid(True, color=C_GRID, linewidth=0.6)
ax.set_axisbelow(True)

legend_handles = [
    mpatches.Patch(color=C_ORANGE, label="Price/m² (dominant)"),
    mpatches.Patch(color=C_NAVY,   label="Categorical location/type"),
    mpatches.Patch(color=C_TEAL,   label="SVD text / numeric"),
]
ax.legend(handles=legend_handles, fontsize=9, loc="lower right",
          frameon=True, framealpha=0.95)

plt.tight_layout(pad=0.8)
save(fig, "fig_featimp.png")

# ══════════════════════════════════════════════════════════════
# FIG 8 — Price segment analysis
# ══════════════════════════════════════════════════════════════
print("Fig 8: fig_price_seg")

seg_labels = ["<1B", "1-3B", "3-10B", "10-50B", ">50B"]
seg_total  = [651, 1953, 3491, 2497, 568]
seg_sus_pct= [39.5, 5.5, 4.0, 14.7, 13.2]
seg_f1     = [0.928, 0.723, 0.667, 0.794, 0.589]

seg_sus_n  = [round(t * p/100) for t, p in zip(seg_total, seg_sus_pct)]
seg_norm_n = [t - s for t, s in zip(seg_total, seg_sus_n)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.2))
fig.patch.set_facecolor("white")
fig.suptitle(f"Price segment analysis (n={n_total:,})",
             fontsize=12, fontweight="bold", y=0.98)

x = np.arange(len(seg_labels))

# Left: stacked bar
ax1.bar(x, seg_norm_n, width=0.55, color=C_NAVY,   label="Normal",     linewidth=0)
ax1.bar(x, seg_sus_n,  width=0.55, color=C_ORANGE, label="Suspicious",
        bottom=seg_norm_n, linewidth=0)

for i, (tot, pct) in enumerate(zip(seg_total, seg_sus_pct)):
    ax1.text(i, tot + 35, f"{pct}%", ha="center", va="bottom",
             fontsize=9.5, fontweight="bold", color=C_TEXT)

ax1.set_xticks(x); ax1.set_xticklabels(seg_labels, fontsize=10)
ax1.set_ylabel("# ads", fontsize=10)
ax1.set_title("Distribution & suspicious\nrate by price segment", fontsize=10)
ax1.set_ylim(0, max(seg_total) * 1.22)
ax1.spines[["top", "right"]].set_visible(False)
ax1.yaxis.grid(True, color=C_GRID, linewidth=0.5); ax1.set_axisbelow(True)
ax1.legend(loc="upper left", fontsize=9.5, frameon=True, framealpha=0.95,
           handlelength=1.2, borderpad=0.5)

# Right: F1 bar
ax2.bar(x, seg_f1, width=0.55, color=C_NAVY, linewidth=0)
ax2.axhline(seg_f1[0], color=C_GRAY, lw=1.2, ls="--", zorder=0)

for i, v in enumerate(seg_f1):
    if v == 0:
        ax2.text(i, 0.02, "0", ha="center", va="bottom",
                 fontsize=10, fontweight="bold", color=C_RED)
    else:
        ax2.text(i, v + 0.018, f"{v:.3f}", ha="center", va="bottom",
                 fontsize=9.5, fontweight="bold", color=C_TEXT)

ax2.set_xticks(x); ax2.set_xticklabels(seg_labels, fontsize=10)
ax2.set_ylabel(r"$F_1$ (M3)", fontsize=10)
ax2.set_title(r"M3 $F_1$ per price segment" + "\n(CV 5-fold)", fontsize=10)
ax2.set_ylim(0, 1.05)
ax2.spines[["top", "right"]].set_visible(False)
ax2.yaxis.grid(True, color=C_GRID, linewidth=0.5); ax2.set_axisbelow(True)

plt.tight_layout(rect=[0, 0, 1, 0.96], pad=1.2)
save(fig, "fig_price_seg.png")

# ══════════════════════════════════════════════════════════════
print("\nAll figures done. Check:", OUT_DIR)